# Does vertically sheared background flow explain measured tilt?

Primary hypothesis: measured `TiltDir` aligns with the trailing integral of surface-minus-depth environmental velocity, and `TiltDis` increases with the magnitude of that accumulated displacement.

This is tested separately for AE/CE, on/off shelf, background definition, depth, and accumulation window. Instantaneous alignment is descriptive; accumulated shear is the mechanistic test because tilt is a displacement.


In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

HERE = Path.cwd()
ANALYSIS_ROOT = HERE.parent
if str(ANALYSIS_ROOT) not in sys.path:
    sys.path.insert(0, str(ANALYSIS_ROOT))
if str(ANALYSIS_ROOT / "beta_effect_background_flow") not in sys.path:
    sys.path.insert(0, str(ANALYSIS_ROOT / "beta_effect_background_flow"))

import seacofs_tilt_tools as tilt
import mechanism_tools as mech

paths = tilt.Paths()
grid = tilt.load_grid(paths.grid, paths.z_r)
df, _ = tilt.load_tilt_tables(paths)
df = mech.require_tilt_measurements(df)
print(f"Rows: {len(df):,}; measured tilts: {df.TiltDis.notna().sum():,}; eddies: {df.Eddy.nunique():,}")


In [ ]:
from beta_effect_background_flow.background_flow_tools import BackgroundConfig, load_background_cache

config = BackgroundConfig()
background = load_background_cache(config)
background = background.drop(columns=[c for c in ["ic", "jc", "month"] if c in background], errors="ignore")
data = mech.merge_one_to_one_or_many_to_one(df, background)
data = mech.add_background_shear(data)
data = tilt.add_pv_gradient_terms(data, grid)
data = tilt.add_region_labels(data, grid)
data = mech.add_topographic_regimes(data, shelf_depth=2000.0)
data[["TiltDis", "TiltDir", "ann_500_shear_mag_ms", "ann_500_tilt_shear_offset"]].describe()


In [ ]:
offset_columns = [f"{method}_tilt_shear_offset" for method in mech.BACKGROUND_PAIRS]
instantaneous = mech.circular_offset_summary(data, offset_columns, group=("Cyc", "ShelfRegime"))
display(instantaneous.sort_values(["ShelfRegime", "metric", "Cyc"]))


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8), sharex=True, sharey=True)
bins = np.arange(-180, 181, 15)
for ax, ((cyc, shelf), part) in zip(axes.flat, data.groupby(["Cyc", "ShelfRegime"])):
    eddy_offsets = part.groupby("Eddy")["ann_500_tilt_shear_offset"].apply(tilt.circular_mean_deg_true_north)
    ax.hist(eddy_offsets, bins=bins, density=True)
    ax.axvline(0, color="black", ls="--")
    ax.set(title=f"{cyc}: {shelf}", xlabel="Tilt − shear direction (degrees)")
plt.tight_layout()


In [ ]:
for method in mech.BACKGROUND_PAIRS:
    data = mech.add_accumulated_shear(data, method=method, windows=(5, 10, 20, 30))

accum_offsets = [c for c in data if c.endswith("d_offset")]
accum_summary = mech.circular_offset_summary(data, accum_offsets, group=("Cyc", "ShelfRegime"))
display(accum_summary.sort_values("resultant_length", ascending=False).head(30))


In [ ]:
# Component-level test: a mechanism must predict both direction and distance.
import statsmodels.formula.api as smf

rows = []
for (cyc, shelf), part in data.groupby(["Cyc", "ShelfRegime"]):
    for window in (5, 10, 20, 30):
        prefix = f"ann_500_accum_{window}d"
        use = part.dropna(subset=["tilt_east_km", "tilt_north_km", f"{prefix}_east_km", f"{prefix}_north_km"])
        for component in ("east", "north"):
            fit = smf.gee(
                f"tilt_{component}_km ~ {prefix}_{component}_km",
                groups="Eddy", data=use,
            ).fit()
            term = f"{prefix}_{component}_km"
            rows.append({"Cyc": cyc, "ShelfRegime": shelf, "window": window,
                         "component": component, "n": len(use), "beta": fit.params[term],
                         "ci_low": fit.conf_int().loc[term, 0], "ci_high": fit.conf_int().loc[term, 1]})
component_tests = pd.DataFrame(rows)
display(component_tests)


## Decision rule

Support for differential advection requires convergence of: (1) offsets concentrated near 0°, (2) positive component slopes, (3) improved results for a physically plausible trailing window, and (4) a within-eddy relationship. Treat on-shelf annulus results cautiously because coastal truncation can bias the sampled current offshore.
